# Real Estate Price Prediction (Tayara)

This notebook does:
1. Load and inspect the dataset
2. Clean and prepare data
3. Run exploratory data analysis (EDA)
4. Train and evaluate a price prediction model
5. Save the trained model for later inference

In [5]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("data/tayara.csv")
MODEL_PATH = Path("artifacts/price_model.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", DATA_PATH.exists())

Dataset exists: True


In [6]:
# Load data (robust parser to handle possible malformed rows)
df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip"
)

print("Shape:", df.shape)
display(df.head(3))
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(20))

Shape: (9296, 9)


,price,transaction,title,city,region,description,surface,bathrooms,rooms
0,6.000000e+02,rent,à RTIBA pour toute saison : la plus belle plag...,Nabeul,Takelsa,Pour groupes et familles (8 personnes et plus)...,3000,2,3
1,1.020000e+09,sale,une villa àvec piscine à vendre,Ariana,La_Soukra,Cette villa de charme est située dans un quart...,270,2,3
2,4.900000e+05,sale,S+3 avec jardin à chotrana 1,Ariana,Chotrana_1,Cet appartement est situé au rez-de-chaussée d...,141,3,3


Shape: (9296, 9)


,price,transaction,title,city,region,description,surface,bathrooms,rooms
0,6.000000e+02,rent,à RTIBA pour toute saison : la plus belle plag...,Nabeul,Takelsa,Pour groupes et familles (8 personnes et plus)...,3000,2,3
1,1.020000e+09,sale,une villa àvec piscine à vendre,Ariana,La_Soukra,Cette villa de charme est située dans un quart...,270,2,3
2,4.900000e+05,sale,S+3 avec jardin à chotrana 1,Ariana,Chotrana_1,Cet appartement est situé au rez-de-chaussée d...,141,3,3



Columns:
['price', 'transaction', 'title', 'city', 'region', 'description', 'surface', 'bathrooms', 'rooms']

Missing values:
price          39
description    15
transaction     0
city            0
title           0
region          0
surface         0
bathrooms       0
rooms           0
dtype: int64


In [ ]:
# Basic cleaning and type normalization
work = df.copy()

# Ensure expected columns exist (create if missing)
for c in ["title", "description", "city", "region", "transaction", "surface", "bathrooms", "rooms", "price"]:
    if c not in work.columns:
        work[c] = np.nan

# Convert numeric-like columns
for c in ["price", "surface", "bathrooms", "rooms"]:
    work[c] = pd.to_numeric(work[c], errors="coerce")

# Text normalization
for c in ["title", "description", "city", "region", "transaction"]:
    work[c] = work[c].astype("string").fillna("").str.strip()

# Remove clearly invalid rows
work = work.dropna(subset=["price"]).copy()
work = work[work["price"] > 0].copy()

# Optional outlier clipping for stable training
q_low, q_high = work["price"].quantile([0.01, 0.99])
work = work[(work["price"] >= q_low) & (work["price"] <= q_high)].copy()

# Merge text columns for a richer text signal
work["text"] = (work["title"].fillna("") + " " + work["description"].fillna(" ")).str.strip()

print("Cleaned shape:", work.shape)
print("Price range:", (work["price"].min(), work["price"].max()))
display(work[["price", "transaction", "city", "region", "surface", "bathrooms", "rooms"]].head(3))

In [ ]:
# EDA: target and key feature distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(work["price"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Price distribution")
axes[0].set_xlabel("price")

sns.histplot(np.log1p(work["price"]), bins=50, kde=True, ax=axes[1], color="orange")
axes[1].set_title("log(1 + price) distribution")
axes[1].set_xlabel("log1p(price)")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.scatterplot(data=work, x="surface", y="price", alpha=0.4, ax=axes[0])
axes[0].set_title("Price vs Surface")

city_stats = work.groupby("city", dropna=False)["price"].median().sort_values(ascending=False).head(10)
sns.barplot(x=city_stats.values, y=city_stats.index, ax=axes[1])
axes[1].set_title("Top 10 cities by median price")
axes[1].set_xlabel("median price")
axes[1].set_ylabel("city")

plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
feature_cols = ["text", "transaction", "city", "region", "surface", "bathrooms", "rooms"]
target_col = "price"

X = work[feature_cols].copy()
y = work[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

In [ ]:
# Preprocessing + model pipeline
numeric_features = ["surface", "bathrooms", "rooms"]
categorical_features = ["transaction", "city", "region"]
text_feature = "text"

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=2,
        strip_accents="unicode"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
        ("txt", text_pipe, text_feature),
    ],
    remainder="drop"
)

base_model = Ridge(alpha=2.0, random_state=42)

model = Pipeline([
    ("prep", preprocessor),
    ("reg", TransformedTargetRegressor(
        regressor=base_model,
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

In [ ]:
# Quick prediction sanity check
sample = X_test.head(5).copy()
sample["true_price"] = y_test.head(5).values
sample["pred_price"] = model.predict(X_test.head(5))
sample[["transaction", "city", "region", "surface", "rooms", "bathrooms", "true_price", "pred_price"]]

In [ ]:
# Save model and useful metadata
artifact = {
    "model": model,
    "feature_columns": feature_cols,
    "target": target_col,
    "price_clip_quantiles": (float(q_low), float(q_high)),
}

joblib.dump(artifact, MODEL_PATH)
print("Saved to:", MODEL_PATH.resolve())